# Lab 3: ID3 Decision Tree

This notebook is designed to be reusable.

- By default it loads a dataset from `data/play_tennis.csv` (relative to this notebook).
- To use your own dataset, set `DATA_PATH` to your CSV path.
- The **target (class) column** is `TARGET_COL` (if not set, the last column is used).

Note: This simple ID3 implementation assumes **categorical** features.


In [ ]:
import pandas as pd
from pathlib import Path
from math import log2

In [ ]:
def entropy(data: pd.DataFrame, target_col: str) -> float:
    target = data[target_col]
    counts = target.value_counts()
    total = len(target)
    ent = 0.0
    for count in counts:
        prob = count / total
        ent += -prob * log2(prob)
    return ent


In [ ]:
def information_gain(data: pd.DataFrame, feature: str, target_col: str) -> float:
    total_entropy = entropy(data, target_col)
    values = data[feature].dropna().unique()
    weighted_entropy = 0.0
    for value in values:
        subset = data[data[feature] == value]
        weighted_entropy += (len(subset) / len(data)) * entropy(subset, target_col)
    return total_entropy - weighted_entropy


In [ ]:
def id3(data: pd.DataFrame, features: list[str], target_col: str):
    # If all target values are the same, return the class
    if len(data[target_col].unique()) == 1:
        return data[target_col].iloc[0]

    # If no features are left, return the majority class
    if not features:
        return data[target_col].mode()[0]

    # Find the feature with the highest information gain
    gains = {feature: information_gain(data, feature, target_col) for feature in features}
    best_feature = max(gains, key=gains.get)

    # Create a tree node for the best feature
    tree = {best_feature: {}}

    # Split the dataset and recursively build the tree for each branch
    for value in data[best_feature].dropna().unique():
        subset = data[data[best_feature] == value]
        remaining_features = [f for f in features if f != best_feature]
        tree[best_feature][value] = id3(subset, remaining_features, target_col)

    return tree


In [ ]:
# Load a dataset (CSV)
DATA_PATH = Path("data/play_tennis.csv")  # change this to your own CSV
TARGET_COL = None  # set to a column name (e.g. "PlayTennis"), or leave None to use the last column

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Fallback dataset (so the notebook still runs if the CSV isn't present)
    df = pd.DataFrame(
        {
            "Outlook": [
                "Sunny",
                "Sunny",
                "Overcast",
                "Rain",
                "Rain",
                "Rain",
                "Overcast",
                "Sunny",
                "Sunny",
                "Rain",
                "Sunny",
                "Overcast",
                "Overcast",
                "Rain",
            ],
            "Temperature": [
                "Hot",
                "Hot",
                "Hot",
                "Mild",
                "Cool",
                "Cool",
                "Cool",
                "Mild",
                "Cool",
                "Mild",
                "Mild",
                "Mild",
                "Hot",
                "Mild",
            ],
            "Humidity": [
                "High",
                "High",
                "High",
                "High",
                "Normal",
                "Normal",
                "Normal",
                "High",
                "Normal",
                "Normal",
                "Normal",
                "High",
                "Normal",
                "High",
            ],
            "Wind": [
                "Weak",
                "Strong",
                "Weak",
                "Weak",
                "Weak",
                "Strong",
                "Strong",
                "Weak",
                "Weak",
                "Weak",
                "Strong",
                "Strong",
                "Weak",
                "Strong",
            ],
            "PlayTennis": [
                "No",
                "No",
                "Yes",
                "Yes",
                "Yes",
                "No",
                "Yes",
                "No",
                "Yes",
                "Yes",
                "Yes",
                "Yes",
                "Yes",
                "No",
            ],
        }
    )
    print(f"Note: {DATA_PATH} not found; using built-in sample dataset instead.")

if TARGET_COL is None:
    TARGET_COL = df.columns[-1]

features = [c for c in df.columns if c != TARGET_COL]
decision_tree = id3(df, features, TARGET_COL)

print("Decision Tree:")
print(decision_tree)
